# 02 — Generate and test the synthetic datasets

**Objectives**
1. Generate the clean dataset reproducibly (fixed seed) and inspect it.
2. Prove it is clean: 41 reference data-quality rules must all pass.
3. Inject the 15 catalogued defects and prove the rules find **exactly** those defects — no misses, no false positives.
4. Compute independent expected results (gold answers) and see how corruption changes them.
5. Export CSV + Parquet, write the manifest, and verify file hashes and round-trips.

**Prerequisites:** notebooks 00–01. **Configuration:** defaults from `GenerationConfig` (mirrors `configs/config.example.yaml`). **Runtime:** about 1 minute. **Cloud access:** none — everything here is local.

**What this notebook produces** (same as `python scripts/generate_data.py`):
`data/generated/{clean,corrupted}/{csv,parquet}/*.{csv,parquet}`, `data/manifests/*.json`, `schemas/bigquery/*.json`.

> Outputs saved in this notebook were produced in the build environment. Because the seed is fixed, your run with the pinned package versions should print identical numbers.

In [1]:
# Locate the project root (folder containing pyproject.toml) so the notebook works
# no matter which folder VS Code starts the kernel in, then make `src/` importable.
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Could not find pyproject.toml - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
print("Project root:", root)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)

Project root: /home/claude/analytics-agent
Python: 3.12.3 | executable: /home/claude/.venv/bin/python


In [2]:
import json, time
import pandas as pd
from analytics_agent import schema_registry as sr
from analytics_agent import metrics as m
from analytics_agent.data_generation.generator import GenerationConfig, generate_clean
from analytics_agent.data_generation.reference_checks import run_reference_checks, summarize
from analytics_agent.data_generation.corruption import corrupt, expected_findings, legitimate_cases
from analytics_agent.data_generation.expected_results import build_expected_results
from analytics_agent.data_generation import export
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

## Step 1 — Configuration

All volumes, the seed, the order window and the snapshot time live in one frozen dataclass. Change values here (or use `--scale` on the CLI) to experiment; the defaults are what the gold answers assume.

In [3]:
cfg = GenerationConfig()
cfg.to_dict()

{'seed': 20260701,
 'n_customers': 2000,
 'n_products': 250,
 'n_orders': 15000,
 'start_date': '2025-07-01',
 'end_date': '2026-06-30',
 'as_of': '2026-07-03 00:00:00',
 'customer_history_start': '2023-01-01'}

## Step 2 — Generate the clean dataset

**Checkpoint:** 2,000 customers, 250 products, 15,000 orders, ~32.8k items, ~15.8k payments, ~14.3k shipments, ~2k refunds.

In [4]:
t0 = time.time()
clean = generate_clean(cfg)
print(f"generated in {time.time() - t0:.1f}s")
pd.DataFrame({"rows": {k: len(v) for k, v in clean.items()}})

generated in 10.9s


,rows
customers,2000
products,250
orders,15000
order_items,32823
payments,15804
shipments,14291
refunds,2047


## Step 3 — Inspect the data

Status mixes should look realistic: most orders delivered, ~4.5% cancelled, a handful pending/processing/shipped near the snapshot.

In [5]:
clean["orders"].head(3)

,order_id,customer_id,order_timestamp,order_date,order_status,sales_channel,currency_code,items_subtotal,discount_amount,shipping_amount,tax_amount,order_total,cancelled_timestamp
0,ORD0000001,CUS001941,2025-07-01 00:45:48+00:00,2025-07-01,delivered,web,USD,356.99,0.0,0.0,28.56,385.55,NaT
1,ORD0000002,CUS000913,2025-07-01 00:51:26+00:00,2025-07-01,delivered,mobile_app,USD,471.88,10.8,0.0,29.97,491.05,NaT
2,ORD0000003,CUS000991,2025-07-01 00:56:18+00:00,2025-07-01,delivered,web,USD,177.96,6.5,0.0,11.14,182.60,NaT


In [6]:
pd.concat({
    "order_status": clean["orders"].order_status.value_counts(),
    "payment_status": clean["payments"].payment_status.value_counts(),
    "shipment_status": clean["shipments"].shipment_status.value_counts(),
    "refund_status": clean["refunds"].refund_status.value_counts(),
}).to_frame("rows")

rows
order_status    delivered        14237
                cancelled          684
                shipped             54
                pending_payment     22
                processing           3
payment_status  captured         14526
                failed            1256
                pending             22
shipment_status delivered        14237
                in_transit          54
refund_status   completed         2019
                pending             28

Monthly booked orders — Q4 seasonality is visible (Nov/Dec peak):

In [7]:
b = m.booked_orders(clean)
b.assign(month=pd.to_datetime(b.order_date).dt.strftime("%Y-%m")).groupby("month").order_id.count().to_frame("booked_orders").T

month,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
booked_orders,1012,1105,1056,1210,1467,1721,1023,861,1185,1190,1242,1222


## Step 4 — Schema conformance

**Checkpoint:** column order matches the registry, required columns have no nulls, primary keys are unique.

In [8]:
for name, df in clean.items():
    t = sr.get_table(name)
    assert list(df.columns) == t.column_names, name
    assert not df[t.required_columns].isna().any().any(), f"nulls in required columns of {name}"
    assert not df.duplicated(list(t.primary_key)).any(), f"duplicate PK in {name}"
print("Schema conformance: OK for all 7 tables")

Schema conformance: OK for all 7 tables


## Step 5 — Run the 41 reference rules on the clean data

These pandas rules are the **test oracle**. Stage 5 implements the same rule IDs in BigQuery SQL and is compared against this. **Checkpoint:** 41 pass, 0 fail.

In [9]:
clean_results = run_reference_checks(clean, cfg.as_of_ts)
clean_summary = summarize(clean_results)
print(clean_summary.status.value_counts().to_dict())
assert (clean_summary.status == "pass").all(), clean_summary[clean_summary.status == "fail"]
clean_summary.head(10)

{'pass': 41}


,rule_id,table,severity,status,violations,example_ids,description
0,CUS_PK_UNIQUE,customers,critical,pass,0,,customer_id is unique
1,CUS_REQUIRED_NOT_NULL,customers,high,pass,0,,Required customer columns are not null
2,CUS_REGION_ALLOWED,customers,medium,pass,0,,region in allowed values
3,CUS_EMAIL_UNIQUE,customers,medium,pass,0,,email is unique
4,PRD_PK_UNIQUE,products,critical,pass,0,,product_id is unique
5,PRD_SKU_UNIQUE,products,high,pass,0,,sku (business key) is unique
6,PRD_CATEGORY_ALLOWED,products,medium,pass,0,,category in allowed values
7,PRD_PRICE_VALID,products,high,pass,0,,list_price > 0 and 0 < unit_cost < list_price
8,ORD_PK_UNIQUE,orders,critical,pass,0,,order_id is unique
9,ORD_REQUIRED_NOT_NULL,orders,high,pass,0,,Required order columns are not null


## Step 6 — Legitimate edge cases (must NOT be flagged)

These look odd but are correct. They exist in the clean data and the rules above still passed, which shows the rules don't produce false positives on them.

In [10]:
pd.DataFrame(legitimate_cases(clean))

,case,count,why_legitimate
0,pending_payment_orders,22,Recent orders awaiting payment confirmation; e...
1,failed_then_captured_payments,994,A declined attempt followed by a successful on...
2,cancelled_orders_with_captured_payment_and_ful...,232,Paid then cancelled; an order-level refund ret...
3,cancelled_orders_without_payment,190,Cancelled before any payment attempt.
4,partial_item_refunds,528,Refund below the line's net amount (e.g. goodw...
5,pending_refunds,28,Initiated within the last days of the snapshot...
6,in_transit_shipments,54,No delivered_timestamp yet; excluded from on-t...
7,processing_orders_without_shipment,3,Paid but not yet shipped.
8,customers_without_orders,98,Registered but never purchased.
9,products_never_sold,0,Catalog items with no sales in the window.


## Step 7 — Inject controlled defects

Own seed (777), small disjoint sets of records, each order touched by at most one defect.

In [11]:
corrupted, defects = corrupt(clean, seed=777)
catalog = pd.DataFrame([d.to_dict() for d in defects])[
    ["defect_id", "table", "defect_type", "severity", "affected_count", "detection_rule"]]
catalog

,defect_id,table,defect_type,severity,affected_count,detection_rule
0,D01,customers,duplicate_primary_key,critical,5,CUS_PK_UNIQUE
1,D02,customers,missing_required_value,high,10,CUS_REQUIRED_NOT_NULL
2,D03,products,invalid_category,medium,3,PRD_CATEGORY_ALLOWED
3,D04,products,duplicate_business_key,high,4,PRD_SKU_UNIQUE
4,D05,orders,orphan_foreign_key,high,8,ORD_CUSTOMER_FK
5,D06,orders,invalid_status,medium,6,ORD_STATUS_ALLOWED
6,D07,orders,incorrect_monetary_total,high,10,ORD_TOTAL_CONSISTENT
7,D08,order_items,negative_quantity,high,5,ITM_QUANTITY_RANGE
8,D09,order_items,orphan_foreign_key,high,5,ITM_ORDER_FK
9,D10,order_items,duplicate_rows,critical,12,ITM_PK_UNIQUE


## Step 8 — Do the rules find exactly the catalogued defects?

For every rule: found IDs must equal expected IDs (including documented cascades). **Checkpoint:** zero mismatches → detection rate 100%, false positives 0.

In [12]:
got = run_reference_checks(corrupted, cfg.as_of_ts)
exp = expected_findings(defects)
rows = []
for rule_id in got:
    g, e = got[rule_id], exp.get(rule_id, set())
    rows.append({"rule_id": rule_id, "expected": len(e), "found": len(g),
                 "missed": len(e - g), "false_positive": len(g - e)})
cmp = pd.DataFrame(rows)
print("rules with mismatches:", int(((cmp.missed > 0) | (cmp.false_positive > 0)).sum()))
assert (cmp.missed == 0).all() and (cmp.false_positive == 0).all()
cmp[cmp.expected > 0]

rules with mismatches: 0


,rule_id,expected,found,missed,false_positive
0,CUS_PK_UNIQUE,5,5,0,0
1,CUS_REQUIRED_NOT_NULL,10,10,0,0
3,CUS_EMAIL_UNIQUE,5,5,0,0
5,PRD_SKU_UNIQUE,2,2,0,0
6,PRD_CATEGORY_ALLOWED,3,3,0,0
10,ORD_STATUS_ALLOWED,6,6,0,0
11,ORD_CUSTOMER_FK,8,8,0,0
12,ORD_TOTAL_CONSISTENT,10,10,0,0
13,ORD_SUBTOTAL_MATCHES_ITEMS,12,12,0,0
16,ITM_PK_UNIQUE,12,12,0,0


## Step 9 — Look at one defect in the data

D11 duplicated 15 captured payments with the same `provider_transaction_id` — this is what inflates *total payments received*.

In [13]:
d11 = next(d for d in defects if d.defect_id == "D11")
txn = d11.expected_findings["PAY_TXN_UNIQUE"][0]
corrupted["payments"][corrupted["payments"].provider_transaction_id == txn]

,payment_id,order_id,provider_transaction_id,payment_timestamp,payment_method,payment_status,amount,failure_reason
9468,PAY0009469,ORD0008997,txn_162944dbbf4b7215,2026-01-31 19:20:47+00:00,paypal,captured,462.86,None
15812,PAY9000008,ORD0008997,txn_162944dbbf4b7215,2026-01-31 19:20:49+00:00,paypal,captured,462.86,None


## Step 10 — Independent expected results (gold answers)

Computed with deterministic pandas functions in `src/analytics_agent/metrics.py` — never by an LLM. **Checkpoint:** ≥ 15 cases; G06 → Southwest; reconciliation bridge = 0.00.

In [14]:
exp_clean = build_expected_results(clean)
exp_corrupt = build_expected_results(corrupted)
cases = {c["test_id"]: c for c in exp_clean["cases"]}
print(len(cases), "gold cases")
assert len(cases) >= 15
assert cases["G06"]["expected_result"]["largest_decline_region"] == "Southwest"
assert cases["G07"]["expected_result"]["bridge_difference"] == 0.0
pd.DataFrame(cases["G01"]["expected_result"])

20 gold cases


,month,merch_sales,refunds,net_sales
0,2026-01,253619.45,12538.70,241080.75
1,2026-02,221626.48,15776.01,205850.47
2,2026-03,293383.39,17188.09,276195.30
3,2026-04,316468.57,21726.05,294742.52
4,2026-05,301178.83,12585.95,288592.88
5,2026-06,306701.04,3697.58,303003.46


Reconciliation (G07): why total payments differ from net sales.

In [15]:
pd.Series(cases["G07"]["expected_result"]).to_frame("USD")

,USD
merch_sales_booked,3626923.43
shipping_booked,25380.69
tax_booked,250621.81
captured_on_cancelled_orders,62590.92
captured_payments,3965516.85
pending_payments,7814.95
completed_item_refunds,190306.93
completed_cancellation_refunds,62590.92
pending_refunds_excluded,3366.97
net_sales,3436616.50


## Step 11 — How corruption changes the answers

Same correct metric code, different data. Shows why the agent must report data-quality caveats.

In [16]:
k = {c["test_id"]: c for c in exp_corrupt["cases"]}
impact = pd.DataFrame([
    {"metric": "net sales (G07)", "clean": cases["G07"]["expected_result"]["net_sales"], "corrupted": k["G07"]["expected_result"]["net_sales"]},
    {"metric": "captured payments (G07)", "clean": cases["G07"]["expected_result"]["captured_payments"], "corrupted": k["G07"]["expected_result"]["captured_payments"]},
    {"metric": "late rate (G05)", "clean": cases["G05"]["expected_result"]["late_rate"], "corrupted": k["G05"]["expected_result"]["late_rate"]},
    {"metric": "categories returned (G02)", "clean": len(cases["G02"]["expected_result"]), "corrupted": len(k["G02"]["expected_result"])},
])
impact["difference"] = impact.corrupted - impact.clean
impact

,metric,clean,corrupted,difference
0,net sales (G07),3.436616e+06,3.434077e+06,-2539.3000
1,captured payments (G07),3.965517e+06,3.970813e+06,5295.8900
2,late rate (G05),1.071000e-01,1.069000e-01,-0.0002
3,categories returned (G02),7.000000e+00,1.000000e+01,3.0000


## Step 12 — Known wrong answers the reviewers must catch

In [17]:
pd.DataFrame(exp_clean["known_wrong_answers"])[["id", "mistake", "wrong_value", "correct_value"]]

,id,mistake,wrong_value,correct_value
0,W01,SUM(orders.order_total) after joining order_it...,1.058479e+07,3.902926e+06
1,W02,Reporting captured payments as net sales,3.965517e+06,3.436616e+06
2,W03,Late rate with all shipments (incl. in_transit...,1.067000e-01,1.071000e-01
3,W04,Net sales subtracting pending as well as compl...,3.433250e+06,3.436616e+06
4,W05,AOV as AVG(order_total) (includes tax and ship...,2.730500e+02,2.537400e+02


## Step 13 — Export CSV + Parquet, manifests and BigQuery schema files

Parquet stores money as `DECIMAL(38,9)` (BigQuery NUMERIC) and timestamps as UTC microseconds. CSV writes `YYYY-MM-DD HH:MM:SS UTC` timestamps, two-decimal money and empty strings for NULL.

In [18]:
from pathlib import Path
out, man = Path("data/generated"), Path("data/manifests")
files = export.write_variant(clean, out, "clean") + export.write_variant(corrupted, out, "corrupted")
export.write_bigquery_schemas(Path("schemas/bigquery"))
export.write_json(export.build_manifest(cfg, files, 777), man / "manifest.json")
export.write_json([d.to_dict() for d in defects], man / "defect_catalog.json")
export.write_json(legitimate_cases(clean), man / "legitimate_cases.json")
export.write_json(exp_clean, man / "expected_results_clean.json")
export.write_json(exp_corrupt, man / "expected_results_corrupted.json")
pd.DataFrame(files)[["variant", "table", "format", "rows", "bytes"]].pivot_table(
    index="table", columns=["variant", "format"], values="bytes", aggfunc="sum")

variant        clean         corrupted        
format           csv parquet       csv parquet
table                                         
customers     187776   54180    188176   54307
order_items  1969288  583104   1970310  583351
orders       1627413  512358   1627401  512427
payments     1506197  750301   1507592  750646
products       22744   12171     22749   12233
refunds       196329   83817    196333   83846
shipments    1450742  445695   1450604  445670

## Step 14 — Verify the exported files

**Checkpoint:** hashes match the manifest, Parquet types match the registry, CSV row counts match.

In [19]:
import hashlib, pyarrow.parquet as pq
manifest = json.loads((man / "manifest.json").read_text())
for f in manifest["files"]:
    assert hashlib.sha256(open(f["path"], "rb").read()).hexdigest() == f["sha256"], f["path"]
for name in sr.TABLE_LOAD_ORDER:
    tbl = pq.read_table(out / "clean" / "parquet" / f"{name}.parquet")
    assert tbl.schema.equals(export.arrow_schema(name), check_metadata=False) or all(
        tbl.schema.field(f.name).type == f.type for f in export.arrow_schema(name))
    assert tbl.num_rows == len(clean[name])
    csv_rows = sum(1 for _ in open(out / "clean" / "csv" / f"{name}.csv")) - 1
    assert csv_rows == len(clean[name]), name
print("Export verified:", len(manifest["files"]), "files")
print(json.dumps(json.load(open("schemas/bigquery/orders.json"))[:3], indent=1))

Export verified: 28 files
[
 {
  "name": "order_id",
  "type": "STRING",
  "mode": "REQUIRED",
  "description": "Surrogate order identifier, format ORD0000001."
 },
 {
  "name": "customer_id",
  "type": "STRING",
  "mode": "REQUIRED",
  "description": "FK to customers.customer_id."
 },
 {
  "name": "order_timestamp",
  "type": "TIMESTAMP",
  "mode": "REQUIRED",
  "description": "When the order was placed (UTC)."
 }
]


## Step 15 — Run the automated test suite from the notebook (optional)

Same as running `pytest` in the terminal. Takes ~30–45 s.

In [20]:
import subprocess
res = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(res.stdout[-600:] or res.stderr[-600:])
assert res.returncode == 0

....................................................                  [100%]
55 passed in 27.62s



## Troubleshooting

| Symptom | Likely cause / fix |
|---|---|
| Numbers differ from saved outputs | Different numpy/pandas versions change random streams or rounding; use the pinned `requirements.txt`. |
| `MemoryError` or slow | Use a smaller config: `GenerationConfig(n_orders=3000, n_customers=500, n_products=80)` (gold values will change). |
| `PermissionError` writing files | The CSV is open in Excel/another program; close it and re-run Step 13. |
| Step 8 mismatch after you edited code | You changed a rule or a defect; update the catalog's `expected_findings` deliberately — never loosen the assertion. |

## Summary and next steps
You generated both datasets, proved the clean one passes every rule, proved the corrupted one fails exactly the catalogued rules, computed 20 gold answers plus 5 known wrong answers, and exported verified CSV/Parquet files with BigQuery schema JSON.

**Next (Stage 3):** load `data/generated/clean/parquet/*` into `analytics_agent_dev` and `corrupted/*` into `analytics_agent_corrupted` using `schemas/bigquery/*.json`, then connect from notebook 03.

## References
- BigQuery loading Parquet: https://cloud.google.com/bigquery/docs/loading-data-cloud-storage-parquet
- BigQuery loading CSV: https://cloud.google.com/bigquery/docs/loading-data-cloud-storage-csv
- NumPy random Generator: https://numpy.org/doc/stable/reference/random/generator.html